# TopKPooling on PROTEINS Graph Classification

Graph Classification on PROTEINS (TUDataset): Hierarchical sparse node selection based on projection onto a learnable score vector. This notebook implements the approach with `TopKPooling` inside a `K3TopKNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `TopKPooling` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "TopKPooling on PROTEINS Graph Classification"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS")
train_batch_size = 60
# drop_last=True keeps every training batch at a fixed size. Keras's
# fit()/train_on_batch() runs one internal forward pass with constant-filled
# placeholder tensors to validate the loss pipeline; if the graph-count in
# a batch were only known from real data (e.g. inferred as `batch.max()+1`),
# that placeholder pass would infer a different (wrong) output size and the
# validation pass would raise a spurious shape-mismatch error. With a fixed
# batch size we can instead pass a static `size` to the pooling calls below.
train_loader = DataLoader(dataset[:800], batch_size=train_batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset[800:], batch_size=60)

in_channels = dataset.num_features
num_classes = dataset.num_classes

# 2. TopKPooling Model
class K3TopKNet(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, batch_size):
        super().__init__()
        self.conv1 = k3_layers.GraphConv(in_channels, hidden_channels)
        self.pool1 = k3_layers.TopKPooling(hidden_channels, ratio=0.8)
        self.conv2 = k3_layers.GraphConv(hidden_channels, hidden_channels)
        self.pool2 = k3_layers.TopKPooling(hidden_channels, ratio=0.8)
        self.conv3 = k3_layers.GraphConv(hidden_channels, hidden_channels)
        self.pool3 = k3_layers.TopKPooling(hidden_channels, ratio=0.8)
        self.lin1 = layers.Dense(128, activation="relu")
        self.drop = layers.Dropout(0.5)
        self.lin2 = layers.Dense(64, activation="relu")
        self.lin3 = layers.Dense(out_channels)
        self.batch_size = batch_size

    def call(self, inputs, training=False):
        x, edge_index, batch = inputs["x"], inputs["edge_index"], inputs.get("batch", None)
        size = self.batch_size
        x = ops.relu(self.conv1(x, edge_index))
        x, edge_index, _, batch, perm, score = self.pool1(x, edge_index, batch=batch)
        x1 = ops.concatenate([k3_layers.global_max_pool(x, batch, size=size), k3_layers.global_mean_pool(x, batch, size=size)], axis=-1)

        x = ops.relu(self.conv2(x, edge_index))
        x, edge_index, _, batch, perm, score = self.pool2(x, edge_index, batch=batch)
        x2 = ops.concatenate([k3_layers.global_max_pool(x, batch, size=size), k3_layers.global_mean_pool(x, batch, size=size)], axis=-1)

        x = ops.relu(self.conv3(x, edge_index))
        x, edge_index, _, batch, perm, score = self.pool3(x, edge_index, batch=batch)
        x3 = ops.concatenate([k3_layers.global_max_pool(x, batch, size=size), k3_layers.global_mean_pool(x, batch, size=size)], axis=-1)

        out = x1 + x2 + x3
        out = self.lin1(out)
        out = self.drop(out, training=training)
        out = self.lin2(out)
        return self.lin3(out)

k3_model = K3TopKNet(in_channels, 128, num_classes, batch_size=train_batch_size)

# 3. Model Compilation
compile_kwargs = {
    "optimizer": keras.optimizers.Adam(learning_rate=0.001),
    "loss": keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    "metrics": [keras.metrics.SparseCategoricalAccuracy(name="acc")],
}
if backend == "jax":
    compile_kwargs["jit_compile"] = False

k3_model.compile(**compile_kwargs)

# 4. Generator & Training
def to_np(t, dtype=None):
    if t is None:
        return None
    if isinstance(t, np.ndarray):
        return t.astype(dtype) if dtype else t
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "cpu"):
        t = t.cpu()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            inputs = {
                "x": to_np(batch.x, dtype=np.float32),
                "edge_index": to_np(batch.edge_index, dtype=np.int64),
                "batch": to_np(batch.batch, dtype=np.int64) if hasattr(batch, "batch") else None,
            }
            y = to_np(batch.y, dtype=np.int64).reshape(-1)
            yield inputs, y

print(f"Training K3-Node TopKNet on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")